# Brewing Time!

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from files_utils import get_root_path
import os

root_path = get_root_path()
os.chdir(root_path)

In [31]:
import numpy as np
import pandas as pd
import polar as pl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit

from pathlib import Path

from opt_project.data_processing import is_more_than_one

In [4]:
data_path = Path('data')

#### Gathering our ingredients

In [5]:
df_beers = pd.read_csv(data_path / 'beer_reviews.csv')
if 'index' in df_beers.columns:
    df_beers.drop('index', inplace=True, axis=1)
df_beers.head()

,brewery_id,brewery_name,review_time,review_overall,review_aroma,review_appearance,review_profilename,beer_style,review_palate,review_taste,beer_name,beer_abv,beer_beerid
0,10325,Vecchio Birraio,1234817823,1.5,2.0,2.5,stcules,Hefeweizen,1.5,1.5,Sausa Weizen,5.0,47986
1,10325,Vecchio Birraio,1235915097,3.0,2.5,3.0,stcules,English Strong Ale,3.0,3.0,Red Moon,6.2,48213
2,10325,Vecchio Birraio,1235916604,3.0,2.5,3.0,stcules,Foreign / Export Stout,3.0,3.0,Black Horse Black Beer,6.5,48215
3,10325,Vecchio Birraio,1234725145,3.0,3.0,3.5,stcules,German Pilsener,2.5,3.0,Sausa Pils,5.0,47969
4,1075,Caldera Brewing Company,1293735206,4.0,4.5,4.0,johnmichaelsen,American Double / Imperial IPA,4.0,4.5,Cauldron DIPA,7.7,64883


To upgrade our analysis we could add:

- the beer info 
- country

Since the nature of our data, we'll proceed with a Collaborative Filtering approach.

### Splitting the dataset

We want to maintain the balance between train and test set to avoid a cold start

In [24]:
# Check if there are more than one review per user

mask_mto_r = is_more_than_one(df_beers, 'review_profilename')

df_beers_mto_r = df_beers[mask_mto_r]


# Check if there are more than one review per beer

mask_mto_r_b = is_more_than_one(df_beers_mto_r, 'beer_beerid')

df_beers_mto_r_b = df_beers_mto_r[mask_mto_r_b]

print(f"df_beers.shape:             {df_beers.shape[0]}\n"
      f"df_beers_mto_r.shape:       {df_beers_mto_r.shape[0]}\n"
      f"df_beerss_mto_r_b.shape:    {df_beers_mto_r_b.shape[0]}\n")


df_beers.shape:             1586614
df_beers_mto_r.shape:       1575823
df_beerss_mto_r_b.shape:    1551992



In [57]:
# Create a test dataframe for beer reviews with country information

# Create sample data
# Create controlled sample data where some users have only one review
# and some beers have only been reviewed once
profiles = ['user1', 'user1', 'user1', 'user1', 'user1', 'user2', 'user2', 'user2', 'user2', 'user2', 'user3', 'user3', 'user3', 'user3', 'user3', 'user4', 'user5']
beer_ids = np.array([1002, 1003, 1001, 1004, 1005, 1001, 1004, 1002, 1003, 1006, 1007, 1002, 1003, 1004, 1005, 1001, 1003])
beer_ratings = np.random.uniform(1.0, 5.0, size=17).round(1)
breweries = ['Craft Brewery', 'Hops & Barley', 'Mountain Brew', 'Urban Ales', 'Vintage Fermentation']
breweries = [breweries[i % len(breweries)] for i in range(17)]
countries = ['USA', 'Germany', 'Belgium', 'UK', 'Czech Republic']
countries = [countries[i % len(countries)] for i in range(17)]

# Create the test dataframe
test_df_with_country = pd.DataFrame({
    'profile': profiles,
    'id_beer': beer_ids,
    'beer_rating': beer_ratings,
    'brewery': breweries,
    'country': countries
})

# Shuffle the dataframe
test_df_with_country = test_df_with_country.sample(frac=1, random_state=42).reset_index(drop=True)

# Print the dataframe with country information
test_df_with_country


,profile,id_beer,beer_rating,brewery,country
0,user1,1002,2.7,Craft Brewery,USA
1,user1,1003,3.5,Hops & Barley,Germany
2,user2,1001,3.7,Craft Brewery,USA
3,user4,1001,2.1,Craft Brewery,USA
4,user3,1002,3.4,Hops & Barley,Germany
5,user3,1005,4.2,Vintage Fermentation,Czech Republic
6,user2,1003,3.0,Urban Ales,UK
7,user3,1004,1.8,Urban Ales,UK
8,user1,1001,3.6,Mountain Brew,Belgium
9,user2,1006,3.5,Vintage Fermentation,Czech Republic


In [58]:
test_df_with_country.groupby('profile').size()

profile
user1    5
user2    5
user3    5
user4    1
user5    1
dtype: int64

In [59]:
mask_test_df = is_more_than_one(test_df_with_country, 'profile')

test_df = test_df_with_country[mask_test_df]
test_df.groupby('profile').size()

profile
user1    5
user2    5
user3    5
dtype: int64

In [63]:
def stratified_split_by_user(df:pd.DataFrame, user_col:str ='profile', beer_col:str ='id_beer', test_size:float=0.2, random_state:int =42) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into train and test sets ensuring:
    - Each user with multiple reviews has reviews in both train and test sets
    - Approximately test_size proportion of each user's reviews are in test set
    - Users with only one review stay in training set
    
    Args:
        df: DataFrame with review data
        user_col: Column name for user identifiers
        test_size: Proportion of data for test set
        random_state: Random seed for reproducibility
        
    Returns:
        train_df, test_df: Split dataframes
    """
    # Create copies to avoid modifying the original dataframe
    mask_user = is_more_than_one(df, user_col)
    mask_beer = is_more_than_one(df, beer_col)

    # Adding all the users with only one review and beers with only a review to the training set
    train_df = pd.DataFrame(df[~mask_user | ~mask_beer])
    test_df = pd.DataFrame()
    
    new_df = df[mask_user | mask_beer]

    # Get counts of reviews per user and beer
    user_counts = df[mask_user].value_counts()
    beer_counts = df[mask_beer].value_counts()
    
    # Sample one review per beer
    for beer, count in beer_counts.items():
        beer_data = df[df[beer_col] == beer]
        
        test_df = pd.concat([test_df, beer_data.sample(n=1, random_state=random_state)])
        new_df = new_df[~new_df.index.isin(beer_data.index)]


    # Calculate the new test size
    new_test_size = round(test_size - (1 - ((df.shape[0] - test_df.shape[0]) / df.shape[0])), 2)

    # If the new test size is negative, add all the remaining samples to the test set
    if new_test_size <= 0:
        train_df = pd.concat([train_df, new_df])
        return train_df, test_df
    
    # Process each user
    for user, count in user_counts.items():
        user_data = df[df[user_col] == user]
        
        if count == 1:
            # Users with only one review go to training set
            train_df = pd.concat([train_df, user_data])
        else:
            # For users with multiple reviews, sample new_test_size proportion for test
            n_test = max(1, int(count * new_test_size))  # At least 1 review in test set
            
            # Sample without replacement
            test_samples = user_data.sample(n=n_test, random_state=random_state)
            train_samples = user_data[~user_data.index.isin(test_samples.index)]
            
            train_df = pd.concat([train_df, train_samples])
            test_df = pd.concat([test_df, test_samples])
    
    # Reset indices
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    
    print(f"Training set: {train_df.shape[0]} samples")
    print(f"Test set: {test_df.shape[0]} samples")


    
    return train_df, test_df

# Apply the function to our dataframe
train_df, test_df = stratified_split_by_user(test_df_with_country)


ValueError: operands could not be broadcast together with shapes (17,) (5,) 

In [62]:
print(f"train_df:\n{train_df}\n")
print(f"test_df:\n{test_df}\n")

train_df:
     column1   column2
83  0.801684  0.579619
53  0.116478  0.917543
70  0.200771  0.690726
45  0.675621  0.007334
44  0.162538  0.474435
39  0.262471  0.854806
22  0.106337  0.014331
80  0.325685  0.765972
10  0.370025  0.264472
0   0.732388  0.116459
18  0.076723  0.247400
30  0.616428  0.542998

test_df:
   profile  id_beer  beer_rating               brewery         country
0    user1     1002          2.7         Craft Brewery             USA
1    user1     1003          3.5         Hops & Barley         Germany
2    user2     1001          3.7         Craft Brewery             USA
4    user3     1002          3.4         Hops & Barley         Germany
5    user3     1005          4.2  Vintage Fermentation  Czech Republic
6    user2     1003          3.0            Urban Ales              UK
7    user3     1004          1.8            Urban Ales              UK
8    user1     1001          3.6         Mountain Brew         Belgium
9    user2     1006          3.5  Vintage 

In [ ]:
def split_data(df: pd.DataFrame, col: list, test_size: float=0.2, random_state: int=42) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into train and test sets with the following conditions:
    - The proportion between test and train sets is 0.2 and 0.8 respectively
    - The train and test set have at least one unique value in the columns specified in col
    """

    df_copy = df.copy()

    for c in col:
        df_copy = is_more_than_one(df_copy, c)
    
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    sss.split(df_copy[col[0]], df[col[1]])
    
    # For users with multiple reviews, ensure at least one review in each set
    train_users, test_users = train_test_split(users_with_multiple, test_size=test_size, random_state=random_state)
    
    
    # For users with multiple reviews, select one review for test and rest for train
    test_multiple = pd.concat([df_multiple[df_multiple['review_profilename'] == user].sample(1, random_state=random_state) 
                              for user in test_users])
    
    train_multiple = df_multiple[~df_multiple.index.isin(test_multiple.index)]
    
    
    print(f"Training set: {df_train.shape[0]} samples")
    print(f"Test set: {df_test.shape[0]} samples")
    
    return df_train, df_test

In [7]:
# First, let's drop rows with null values in the stratification columns
df_beers_clean = df_beers.dropna(subset=['review_profilename'])

for profilename in df_beers_clean[]

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

SyntaxError: invalid syntax (1877378715.py, line 4)

Since we have